# Phase5: Factor Combination

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from typing import Dict, Callable
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna

In [22]:
in_dir = Path("data/cache/final_factors")

meta = pd.read_parquet(in_dir / "final_factor_meta.parquet")
X_final = pd.read_parquet(in_dir / "final_factor_panel.parquet")

final_factors = meta.index.tolist()
factor_sign = meta["sign"]
nw_t = meta["nw_t"]

In [23]:
close_path = Path("data/cache/processed_fast/close.parquet")
close_raw = pd.read_parquet(close_path)

In [24]:
import operators_library
import analytics
import ml_utilities

In [25]:
# X_final loaded from parquet, columns MultiIndex (factor, asset)
X_rank = X_final.copy()
X_rank.columns = X_rank.columns.set_names(["factor", "ticker"])
X_rank.index = pd.to_datetime(X_rank.index)

# Use saved sign if available; fallback from nw_t (<0 => invert)
factor_sign = meta["sign"].astype(float).reindex(X_rank.columns.get_level_values("factor").unique()).fillna(1.0)

# Apply sign globally once
X_rank = analytics.apply_sign(X_rank, alpha_panel="factor", sign=factor_sign)

Inverted factors: ['st_4', 'mom_1', 'mom_5']


alpha
st_4    -1.0
mom_1   -1.0
mom_5   -1.0
vo_3     1.0
mr_8     1.0
mr_10    1.0
mr_6     1.0
va_7     1.0
gr_3     1.0
mr_2     1.0
mom_2    1.0
va_4     1.0
Name: sign, dtype: float64

## 5.1 Rebuild Returns + Sample Mask


In [26]:
h = 1                    # forward return horizon
min_factor_frac = 0.5    # require at least this fraction of factors present per (date,ticker)
min_factors_floor = 3    # hard floor
# ----------------------------

# Factors and tickers in this notebook
factors = X_rank.columns.get_level_values("factor").unique().tolist()
tickers = X_rank.columns.get_level_values("ticker").astype(str).unique().tolist()

close_df = analytics.simplify_columns(
    analytics.ensure_datetime_index(
        analytics.to_numeric(close_raw)
    )
)
close_df.columns = close_df.columns.astype(str).str.strip()
close_df = close_df.reindex(columns=tickers)  # align to factor universe tickers

# Align dates
dates = X_rank.index.intersection(close_df.index)
X_rank = X_rank.reindex(index=dates)
close_panel = close_df.reindex(index=dates, columns=tickers).astype(float)

# Tradable universe from price availability
univ = close_panel.notna()

# Factor availability count per (date,ticker)
avail_count = X_rank.notna().T.groupby(level="ticker").sum().T.reindex(index=dates, columns=tickers)
k_min = max(min_factors_floor, int(np.ceil(min_factor_frac * len(factors))))

# Final sample mask (date x ticker)
sample_mask = (univ & (avail_count >= k_min)).fillna(False)

# Forward returns panel (date x ticker)
R_fwd_s = close_panel.pct_change(h, fill_method=None).shift(-h)

print("X_rank shape:", X_rank.shape)
print("close_panel shape:", close_panel.shape)
print("R_fwd_s shape:", R_fwd_s.shape)
print("sample_mask true ratio:", float(sample_mask.values.mean()))
print("k_min factors required:", k_min)

display(R_fwd_s.head())
display(sample_mask.head())

X_rank shape: (4391, 21252)
close_panel shape: (4391, 1771)
R_fwd_s shape: (4391, 1771)
sample_mask true ratio: 0.2741563289522059
k_min factors required: 6


,000005,000006,000008,000009,000012,000016,000021,000025,000027,000028,...,688692,688702,688709,688728,688772,688777,688778,688779,688819,689009
Date,,,,,,,,,,,,,,,,,,,,,
2008-01-29,NaN,0.015326,NaN,NaN,NaN,-0.018182,NaN,NaN,NaN,0.053257,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-30,NaN,-0.018868,NaN,NaN,NaN,-0.003472,NaN,NaN,NaN,-0.063205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-31,NaN,-0.016538,NaN,NaN,NaN,-0.047619,NaN,NaN,NaN,0.000964,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-02-01,NaN,0.075479,NaN,NaN,NaN,0.075610,NaN,NaN,NaN,0.082330,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-02-04,NaN,-0.016364,NaN,NaN,NaN,0.021542,NaN,NaN,NaN,0.006673,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,000005,000006,000008,000009,000012,000016,000021,000025,000027,000028,...,688692,688702,688709,688728,688772,688777,688778,688779,688819,689009
Date,,,,,,,,,,,,,,,,,,,,,
2008-01-29,False,True,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2008-01-30,False,True,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2008-01-31,False,True,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2008-02-01,False,True,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2008-02-04,False,True,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


## 5.2 ICIR Weighted

## Benchmark: Equal Weight

In [27]:
min_n = 30
eps = 1e-12
alpha_level = "factor"

X_rank.columns = X_rank.columns.set_names([alpha_level, "ticker"])

# Equal weights
w_eq = pd.Series(1.0, index=factors, dtype=float)
w_eq = w_eq / (w_eq.abs().sum() + eps)

R_w = R_fwd_s.sort_index()
M_w = sample_mask.sort_index().fillna(False).astype(bool)

dates = X_rank.index.intersection(R_w.index).intersection(M_w.index)

scores = []
for t in dates:
    xrow = X_rank.loc[t]
    s = analytics.score_from_weights(xrow, w_eq, alpha_panel=alpha_level)
    m = M_w.loc[t].reindex(s.index).fillna(False) & R_w.loc[t].reindex(s.index).notna()
    scores.append(s.where(m).rename(t))

S_eq = pd.DataFrame(scores).sort_index()

R_eval = R_w.reindex(index=S_eq.index, columns=S_eq.columns)
ic_eq = analytics.daily_ic(S_eq, R_eval, method="spearman", min_n=min_n)

print("Equal-weight combo IC mean/std/ICIR:",
      float(ic_eq.mean()),
      float(ic_eq.std(ddof=1)),
      float(analytics.icir(ic_eq)))

Equal-weight combo IC mean/std/ICIR: 0.05304244775680666 0.15646948150816622 0.3389954849057153


count    4390.000000
mean        0.053042
std         0.156469
min        -0.741468
25%        -0.046538
50%         0.057686
75%         0.161921
max         0.534456
dtype: float64

## Rolling ICIR

In [31]:
lookback = 256
min_history = 60
min_ic_days = 40

# 1) build per-factor daily IC series
ic_by_alpha = {}
for a in factors:
    s_a = X_rank.xs(a, axis=1, level=alpha_level).reindex(index=dates)
    r_a = R_fwd_s.reindex(index=dates, columns=s_a.columns)
    m_a = sample_mask.reindex(index=dates, columns=s_a.columns).fillna(False)

    ic_a = analytics.daily_ic(
        s_a.where(m_a),
        r_a.where(m_a),
        method="spearman",
        min_n=min_n
    )
    ic_by_alpha[a] = ic_a

# IC statistics Review
ic_table = pd.DataFrame({
    "alpha": factors,
    "n_days": [int(ic_by_alpha[a].notna().sum()) for a in factors],
    "ic_mean": [float(ic_by_alpha[a].mean()) for a in factors],
    "ic_std": [float(ic_by_alpha[a].std(ddof=1)) for a in factors],
    "icir": [float(analytics.icir(ic_by_alpha[a].dropna())) for a in factors],
}).sort_values("icir", ascending=False)
display(ic_table)

# 2) rolling ICIR weights panel
W_icir = pd.DataFrame(index=dates, columns=factors, dtype=float)

for i, t in enumerate(dates):
    past_dates = dates[max(0, i - lookback): i]
    if len(past_dates) < min_history:
        continue

    w = {}
    for a in factors:
        s = ic_by_alpha[a].reindex(past_dates).dropna()
        if len(s) < min_ic_days:
            w[a] = 0.0
            continue

        mu = float(s.mean())
        sd = float(s.std(ddof=1))
        icir = mu / sd if (np.isfinite(sd) and sd > 0) else 0.0
        w[a] = max(0.0, icir)

    w = pd.Series(w, index=factors, dtype=float).fillna(0.0)
    if w.sum() <= eps:
        continue
    w = w / (w.abs().sum() + eps)  
    W_icir.loc[t] = w

W_icir = W_icir.dropna(how="all")
print("ICIR-weight panel shape:", W_icir.shape)
display(W_icir.tail())

# 3) build combined score panel from rolling weights
scores = []
for t in W_icir.index:
    xrow = X_rank.loc[t]             
    w = W_icir.loc[t].astype(float)
    s = analytics.score_from_weights(xrow, w, alpha_panel=alpha_level)

    m = sample_mask.loc[t].reindex(s.index).fillna(False) & R_fwd_s.loc[t].reindex(s.index).notna()
    scores.append(s.where(m).rename(t))

S_icir = pd.DataFrame(scores).sort_index()
print("S_icir shape:", S_icir.shape)

# 4) evaluate combined IC
R_eval = R_fwd_s.reindex(index=S_icir.index, columns=S_icir.columns)
M_eval = sample_mask.reindex(index=S_icir.index, columns=S_icir.columns).fillna(False)

ic_icir_combo = analytics.daily_ic(
    S_icir.where(M_eval),
    R_eval.where(M_eval),
    method="spearman",
    min_n=min_n
)

print("ICIR-weight combo IC mean/std/ICIR:",
      float(ic_icir_combo.mean()),
      float(ic_icir_combo.std(ddof=1)),
      float(analytics.icir(ic_icir_combo)))

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,alpha,n_days,ic_mean,ic_std,icir
0,vo_3,4390,0.045975,0.125356,0.366759
1,mr_8,4390,0.048436,0.137234,0.352947
2,st_4,4390,0.053814,0.158709,0.339075
4,mr_6,4390,0.013098,0.063742,0.205478
6,mom_1,4390,0.024706,0.161308,0.153160
5,va_7,4390,0.011730,0.085499,0.137189
11,va_4,3134,0.006723,0.060436,0.111235
8,mr_2,4390,0.008296,0.087049,0.095299
7,gr_3,4390,0.005521,0.064962,0.084991
3,mr_10,4279,0.004410,0.054681,0.080657


ICIR-weight panel shape: (4331, 12)


,vo_3,mr_8,st_4,mr_10,mr_6,va_7,mom_1,gr_3,mr_2,mom_5,mom_2,va_4
Date,,,,,,,,,,,,
2026-02-13,0.132112,0.204632,0.112318,0.094401,0.143363,0.044130,0.096912,0.0,0.040107,0.066306,0.0,0.065719
2026-02-24,0.133031,0.207144,0.110573,0.093980,0.153185,0.042386,0.096003,0.0,0.038158,0.065949,0.0,0.059591
2026-02-25,0.131875,0.203178,0.114094,0.099115,0.153672,0.045235,0.095438,0.0,0.035945,0.061272,0.0,0.060177
2026-02-26,0.132170,0.204248,0.115949,0.105300,0.151249,0.050432,0.088846,0.0,0.031696,0.060727,0.0,0.059381
2026-02-27,0.132586,0.209625,0.117708,0.107363,0.147412,0.049044,0.087694,0.0,0.031713,0.059319,0.0,0.057537


S_icir shape: (4331, 1771)
ICIR-weight combo IC mean/std/ICIR: 0.053711048487421274 0.1571017732957621 0.34188696512231


count    4330.000000
mean        0.053711
std         0.157102
min        -0.740465
25%        -0.047049
50%         0.058438
75%         0.162581
max         0.553341
dtype: float64

## 5.2 Machine Learning

In [32]:
# Rebuild X_z
alpha_level = "factor"   # current level name in X_rank
ticker_level = "ticker"

# 1) ensure naming convention used downstream
X_tmp = X_rank.copy()
X_tmp.columns = X_tmp.columns.set_names(["factor", "ticker"])

# 2) build per-factor z-scored panel, then stack back to MultiIndex columns
alphas = X_tmp.columns.get_level_values("factor").unique().tolist()

Xz_parts = {}
for a in alphas:
    Xa = X_tmp.xs(a, axis=1, level="factor")     # date x ticker
    Xa_w = analytics._winsorize_by_row(Xa)      # optional but consistent with earlier pipeline
    Xa_z = analytics._zscore_by_row(Xa_w)
    Xz_parts[a] = Xa_z

X_z = pd.concat(Xz_parts, axis=1)
X_z.columns = X_z.columns.set_names(["factor", "ticker"])
X_z = X_z.sort_index().sort_index(axis=1)

print("X_z shape:", X_z.shape)


X_z shape: (4391, 21252)


factor       gr_3                                                         \
ticker     000005    000006 000008 000009 000012    000016 000021 000025   
Date                                                                       
2008-01-29    NaN -1.447868    NaN    NaN    NaN  0.721756    NaN    NaN   
2008-01-30    NaN -1.445881    NaN    NaN    NaN  0.718362    NaN    NaN   
2008-01-31    NaN -1.444982    NaN    NaN    NaN  0.720060    NaN    NaN   
2008-02-01    NaN -1.459283    NaN    NaN    NaN  0.716768    NaN    NaN   
2008-02-04    NaN -1.446702    NaN    NaN    NaN  0.716046    NaN    NaN   

factor                       ...   vo_3                                     \
ticker     000027    000028  ... 688692 688702 688709 688728 688772 688777   
Date                         ...                                             
2008-01-29    NaN  0.046762  ...    NaN    NaN    NaN    NaN    NaN    NaN   
2008-01-30    NaN  0.045042  ...    NaN    NaN    NaN    NaN    NaN    NaN   
2008-01-31    NaN  0.046492  ...    NaN    NaN    NaN    NaN    NaN    NaN   
2008-02-01    NaN  0.039774  ...    NaN    NaN    NaN    NaN    NaN    NaN   
2008-02-04    NaN  0.043191  ...    NaN    NaN    NaN    NaN    NaN    NaN   

factor                                  
ticker     688778 688779 688819 689009  
Date                                    
2008-01-29    NaN    NaN    NaN    NaN  
2008-01-30    NaN    NaN    NaN    NaN  
2008-01-31    NaN    NaN    NaN    NaN  
2008-02-01    NaN    NaN    NaN    NaN  
2008-02-04    NaN    NaN    NaN    NaN  

[5 rows x 21252 columns]

## Linear Methods

In [34]:
data = ml_utilities.panel_to_long(X_z, R_fwd_s, sample_mask,'date','ticker')

alphas_used = [c for c in data.columns if c != "y"]
dates = data.index.get_level_values("date").unique().sort_values()

print("Long panel shape:", data.shape, "(rows x (features + y))")
print("#features:", len(alphas_used), " #dates:", len(dates))

/Users/apple/Documents/GitHub/Systematic-Equities/ml_utilities.py:75: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  X_long = X.stack(level=ticker_column)


Long panel shape: (1022267, 13) (rows x (features + y))
#features: 12  #dates: 2820


In [ ]:
LOOKBACK = 252
MIN_TRAIN_DAYS = 60
MIN_NAMES_PER_DAY = 30

# Define model builders
models: Dict[str, Callable[[], object]] = {
    "OLS": lambda: LinearRegression(),
    "Ridge(1e-2)": lambda: Ridge(alpha=1e-2),
    "Ridge(1e-1)": lambda: Ridge(alpha=1e-1),
    "Ridge(1.0)":  lambda: Ridge(alpha=1.0),
    "ElasticNet(0.01,0.5)": lambda: ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
    "Lasso(0.001)": lambda: Lasso(alpha=0.001, max_iter=20000),
    "Huber": lambda: HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=2000),
}

results = []

y_all = data["y"]

for name, builder in models.items():
    pred = ml_utilities.walkforward_predict(data, 'date', builder, lookback=LOOKBACK, min_train_days=MIN_TRAIN_DAYS,)
    if pred.empty:
        continue

    ic = ml_utilities.daily_ic_long(pred, y_all,'date')
    ls = analytics.daily_ls_spread(pred, y_all, q=0.2)

    row = {"model": name}
    row.update(analytics.summarize_series(ic, "ic"))
    row.update(analytics.summarize_series(ls, "ls"))
    results.append(row)

res = pd.DataFrame(results).sort_values("ic_ir", ascending=False)
display(res)

In [44]:
# Build S_huber
pred_huber = ml_utilities.walkforward_predict(
    data=data,
    date_column="date",
    model_builder=lambda: HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=2000),
    lookback=252,
    min_train_days=60,
    min_n=30,
)

# wide score panel
S_huber = pred_huber.unstack("ticker").sort_index()

# align + mask to tradable/evaluable universe
common_dates = S_huber.index.intersection(sample_mask.index).intersection(R_fwd_s.index)
common_tickers = S_huber.columns.intersection(sample_mask.columns).intersection(R_fwd_s.columns)

S_huber = S_huber.reindex(index=common_dates, columns=common_tickers)
M_eval = sample_mask.reindex(index=common_dates, columns=common_tickers).fillna(False)
R_eval = R_fwd_s.reindex(index=common_dates, columns=common_tickers)

S_huber = S_huber.where(M_eval & R_eval.notna())

print("S_huber shape:", S_huber.shape)
display(S_huber.head())

S_huber shape: (2760, 1253)


,000008,000009,000012,000021,000025,000027,000028,000030,000031,000032,...,688567,688578,688617,688690,688772,688777,688778,688779,688819,689009
2013-08-30,NaN,NaN,NaN,NaN,NaN,NaN,-0.001283,NaN,0.000277,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-09-02,NaN,NaN,NaN,NaN,NaN,NaN,-0.002094,NaN,0.000634,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-09-03,NaN,NaN,NaN,NaN,NaN,NaN,-0.001230,NaN,0.000103,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-09-04,NaN,NaN,NaN,NaN,NaN,NaN,-0.001335,NaN,0.001278,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-09-05,NaN,NaN,NaN,NaN,NaN,NaN,-0.001231,NaN,0.001076,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
